[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/098.%20The%20Green%20Vehicle%20Routing%20Problem/P98-Tier-6_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 98. The Green Vehicle Routing Problem

## Tier 6: Multi-Agent Negotiation

### Purpose and scope
This notebook demonstrates how autonomous vehicle agents can allocate delivery tasks through the Contract Net Protocol (CNP). Each task is announced, eligible agents submit bids, and the task is awarded to the bidder with the highest utility score. A simulated battery-depletion event then illustrates task reassignment.

The example uses four vehicles (two diesel and two electric) and five delivery tasks. It is a small educational simulation of decentralized negotiation, not a vehicle-routing optimizer: it does not guarantee globally optimal routes or compare its results with a centralized optimizer.

### Model assumptions
- Locations are nodes in a fixed distance matrix; distances are measured in kilometres.
- Each awarded task adds one direct leg from an agent's current location to the task location. The agent's location, available fuel or battery, and simulated time are updated after an award.
- Vehicle capacity is the total demand of tasks assigned to that agent. This simplified model does not simulate unloading or a return trip to the depot.
- The operating day starts at 08:00. A time window specifies the earliest and latest permitted service-start hour. If a vehicle arrives early, it waits until the window opens.
- Diesel fuel is measured in litres, electric energy in kWh, and emissions are operational emissions estimated from the vehicle's emissions factor. Electric vehicles have zero tailpipe emissions in this example; electricity-generation emissions are outside the model.
- Task deadlines are checked against the simulated completion time. The simulation does not model traffic, charging stops, stochastic travel times, or communication failures.

### How the Contract Net Protocol works
A task announcer broadcasts a task. Each other agent checks whether it can serve the task, then calculates a bid containing cost, emissions, required time, confidence, and a utility score. The task is awarded to the feasible bidder with the greatest score. Awards are processed task by task so later bids account for the agent's updated load, location, resources, and time.

For agent $i$ and task $j$, the bid utility is

$$
U_{ij} = w_i^{c}\frac{1}{1+C_{ij}} + w_i^{e}\frac{1}{1+E_{ij}} + w_i^{t}\frac{1}{1+T_{ij}} + w_i^{q}Q_{ij}.
$$

Here, $C_{ij}$ is marginal monetary cost, $E_{ij}$ is estimated operational emissions, $T_{ij}$ is travel, waiting, and service time in hours, and $Q_{ij}$ is a confidence score between zero and one. The non-negative weights $w_i^{c}, w_i^{e}, w_i^{t}, w_i^{q}$ express agent $i$'s preferences and sum to one in this example. The reciprocal terms make lower costs, emissions, and times more desirable; the confidence term rewards bids with more remaining resources. The score is a teaching model, not a monetary auction price or a guarantee of system-wide optimality.

### Negotiation sequence
1. Define vehicle agents, tasks, locations, and their operational data.
2. For each task, collect feasible bids from agents other than its announcer.
3. Rank bids by utility and award the task to the highest-scoring bidder, if one exists.
4. Simulate a battery constraint and announce a failed task for reassignment.
5. Inspect agent workloads, negotiation records, and system metrics.

### Interpreting the results
The assignments show how these particular utility weights and feasibility rules influence local decisions. They do not establish global optimality, general resilience, or a universal scaling law. Whether a task can be reassigned depends on the other agents' remaining capacity, energy, time windows, and deadlines. The visualizations summarize this run; they are descriptive rather than a comparison with another optimization method.

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

# Fixed seeds keep any future randomized extensions reproducible.
np.random.seed(42)
random.seed(42)

# All tasks and agents share this simulated operating-day clock.
simulation_start = datetime.now().replace(
    hour=8, minute=0, second=0, microsecond=0
)

In [ ]:
class TaskType(Enum):
    DELIVERY = "delivery"
    PICKUP = "pickup"
    CHARGING = "charging"
    MAINTENANCE = "maintenance"


@dataclass
class Task:
    """A service task announced to the agent market."""

    id: int
    task_type: TaskType
    location: int
    demand: float
    time_window: Tuple[int, int]
    service_time: float
    priority: int
    deadline: datetime
    announced_by: int
    awarded_to: Optional[int] = None
    bids: List["Bid"] = field(default_factory=list)


@dataclass
class Bid:
    """An agent's feasible offer to complete a task."""

    agent_id: int
    task_id: int
    cost: float
    emissions: float
    time_required: float
    confidence: float
    utility_score: float = 0.0


@dataclass
class VehicleAgent:
    """Vehicle state and local bid calculations for one autonomous agent."""

    id: int
    vehicle_type: str
    capacity: float
    current_location: int
    current_fuel: float
    max_fuel: float
    fuel_consumption_rate: float
    emissions_factor: float
    current_tasks: List[Task]
    completed_tasks: List[Task]
    status: str
    utility_weights: Dict[str, float]
    current_time: datetime

    def calculate_marginal_cost(self, task: Task, distances: np.ndarray) -> float:
        """Estimate fuel or electricity cost plus driver time cost."""
        distance = distances[self.current_location, task.location]
        energy_needed = distance * self.fuel_consumption_rate

        if self.vehicle_type == "electric":
            energy_cost = energy_needed * 0.15
        else:
            energy_cost = energy_needed * 1.50

        travel_time = distance / 50.0
        total_time = self.calculate_time_required(task, distances)
        waiting_and_service_time = total_time - travel_time
        driver_cost = total_time * 20.0
        return energy_cost + driver_cost

    def calculate_marginal_emissions(
        self, task: Task, distances: np.ndarray
    ) -> float:
        """Estimate operational emissions for the direct travel leg."""
        distance = distances[self.current_location, task.location]
        energy_needed = distance * self.fuel_consumption_rate
        return energy_needed * self.emissions_factor

    def calculate_time_required(
        self, task: Task, distances: np.ndarray
    ) -> float:
        """Include travel, any wait for the window, and service time."""
        distance = distances[self.current_location, task.location]
        arrival_time = self.current_time + timedelta(hours=distance / 50.0)
        day_start = self.current_time.replace(
            hour=0, minute=0, second=0, microsecond=0
        )
        window_start = day_start + timedelta(hours=task.time_window[0])
        service_start = max(arrival_time, window_start)
        elapsed_time = (service_start - self.current_time).total_seconds() / 3600
        return elapsed_time + task.service_time

    def can_handle_task(self, task: Task, distances: np.ndarray) -> bool:
        """Check energy, cumulative capacity, service window, and deadline."""
        distance = distances[self.current_location, task.location]
        energy_needed = distance * self.fuel_consumption_rate
        if energy_needed > self.current_fuel:
            return False

        current_load = sum(existing.demand for existing in self.current_tasks)
        if current_load + task.demand > self.capacity:
            return False

        arrival_time = self.current_time + timedelta(hours=distance / 50.0)
        day_start = self.current_time.replace(
            hour=0, minute=0, second=0, microsecond=0
        )
        window_start = day_start + timedelta(hours=task.time_window[0])
        window_close = day_start + timedelta(hours=task.time_window[1])
        service_start = max(arrival_time, window_start)

        if service_start > window_close:
            return False

        completion_time = service_start + timedelta(hours=task.service_time)
        return completion_time <= task.deadline

    def submit_bid(self, task: Task, distances: np.ndarray) -> Optional[Bid]:
        """Return a utility-scored bid only when the task is feasible."""
        if not self.can_handle_task(task, distances):
            return None

        cost = self.calculate_marginal_cost(task, distances)
        emissions = self.calculate_marginal_emissions(task, distances)
        time_required = self.calculate_time_required(task, distances)
        distance = distances[self.current_location, task.location]
        energy_needed = distance * self.fuel_consumption_rate
        current_load = sum(existing.demand for existing in self.current_tasks)

        energy_confidence = 1.0 - energy_needed / self.current_fuel
        capacity_confidence = 1.0 - (
            (current_load + task.demand) / self.capacity
        )
        confidence = (energy_confidence + capacity_confidence) / 2.0

        bid = Bid(
            agent_id=self.id,
            task_id=task.id,
            cost=cost,
            emissions=emissions,
            time_required=time_required,
            confidence=confidence,
        )
        bid.utility_score = (
            self.utility_weights["cost"] / (1.0 + cost)
            + self.utility_weights["emissions"] / (1.0 + emissions)
            + self.utility_weights["time"] / (1.0 + time_required)
            + self.utility_weights["confidence"] * confidence
        )
        return bid


class ContractNetProtocol:
    """Task announcement, bidding, evaluation, and contract awarding."""

    def __init__(self, agents: List[VehicleAgent]):
        self.agents = agents
        self.agents_by_id = {agent.id: agent for agent in agents}
        self.task_market: List[Task] = []
        self.contracts: List[Task] = []
        self.negotiation_history: List[Dict] = []
        self.distances = np.array(
            [
                [0, 10, 15, 12, 18, 8],
                [10, 0, 8, 14, 12, 15],
                [15, 8, 0, 10, 6, 12],
                [12, 14, 10, 0, 8, 10],
                [18, 12, 6, 8, 0, 14],
                [8, 15, 12, 10, 14, 0],
            ],
            dtype=float,
        )

    def announce_task(self, task: Task) -> None:
        """Broadcast a task and collect bids from eligible other agents."""
        self.task_market.append(task)
        for agent in self.agents:
            if agent.id != task.announced_by:
                bid = agent.submit_bid(task, self.distances)
                if bid is not None:
                    task.bids.append(bid)

        self.negotiation_history.append(
            {
                "timestamp": datetime.now(),
                "action": "task_announced",
                "task_id": task.id,
                "announced_by": task.announced_by,
                "bids_received": len(task.bids),
            }
        )

    def evaluate_bids(self, task: Task) -> Optional[Bid]:
        """Select the feasible bid with the highest utility score."""
        if not task.bids:
            return None

        task.bids.sort(key=lambda bid: bid.utility_score, reverse=True)
        best_bid = task.bids[0]
        self.negotiation_history.append(
            {
                "timestamp": datetime.now(),
                "action": "bid_evaluated",
                "task_id": task.id,
                "winner": best_bid.agent_id,
                "utility_score": best_bid.utility_score,
                "total_bids": len(task.bids),
            }
        )
        return best_bid

    def award_contract(self, task: Task, bid: Bid) -> None:
        """Commit the task and update the winner's simulated vehicle state."""
        winning_agent = self.agents_by_id[bid.agent_id]
        distance = self.distances[
            winning_agent.current_location, task.location
        ]
        energy_used = distance * winning_agent.fuel_consumption_rate

        winning_agent.current_tasks.append(task)
        winning_agent.current_fuel -= energy_used
        winning_agent.current_location = task.location
        winning_agent.current_time += timedelta(hours=bid.time_required)
        task.awarded_to = bid.agent_id

        if task in self.task_market:
            self.task_market.remove(task)
        self.contracts.append(task)
        self.negotiation_history.append(
            {
                "timestamp": datetime.now(),
                "action": "contract_awarded",
                "task_id": task.id,
                "awarded_to": bid.agent_id,
                "contract_value": bid.cost,
            }
        )

    def handle_dynamic_reassignment(
        self, agent_id: int, failed_task: Task
    ) -> None:
        """Reannounce a failed task and award it if a feasible bid exists."""
        print(
            f"Agent {agent_id} cannot complete task {failed_task.id}; "
            "the task is being reassigned."
        )
        new_task = Task(
            id=failed_task.id + 1000,
            task_type=failed_task.task_type,
            location=failed_task.location,
            demand=failed_task.demand,
            time_window=failed_task.time_window,
            service_time=failed_task.service_time,
            priority=1,
            deadline=failed_task.deadline,
            announced_by=agent_id,
        )

        self.announce_task(new_task)
        best_bid = self.evaluate_bids(new_task)
        if best_bid is not None:
            self.award_contract(new_task, best_bid)
            print(f"Task reassigned to Agent {best_bid.agent_id}.")
        else:
            if new_task in self.task_market:
                self.task_market.remove(new_task)
            print("No agent can feasibly accept the reassigned task.")

    def run_negotiation_round(self, tasks: List[Task]) -> Dict:
        """Process each task through the CNP before moving to the next."""
        print(f"\nStarting negotiation round with {len(tasks)} tasks...")
        results = {
            "tasks_announced": 0,
            "contracts_awarded": 0,
            "failed_assignments": 0,
            "total_cost": 0.0,
            "total_emissions": 0.0,
            "agent_utilization": {},
        }

        for task in tasks:
            self.announce_task(task)
            results["tasks_announced"] += 1
            best_bid = self.evaluate_bids(task)
            if best_bid is None:
                results["failed_assignments"] += 1
                if task in self.task_market:
                    self.task_market.remove(task)
                continue

            self.award_contract(task, best_bid)
            results["contracts_awarded"] += 1
            results["total_cost"] += best_bid.cost
            results["total_emissions"] += best_bid.emissions

        for agent in self.agents:
            results["agent_utilization"][agent.id] = len(agent.current_tasks)
        return results

    def get_negotiation_summary(self) -> pd.DataFrame:
        """Return the recorded announcement, evaluation, and award events."""
        return pd.DataFrame(self.negotiation_history)

In [ ]:
print("=" * 60)
print("MULTI-AGENT NEGOTIATION SYSTEM SETUP")
print("=" * 60)

agents = [
    VehicleAgent(
        id=0,
        vehicle_type="diesel",
        capacity=25,
        current_location=0,
        current_fuel=80,
        max_fuel=80,
        fuel_consumption_rate=0.08,
        emissions_factor=2.68,
        current_tasks=[],
        completed_tasks=[],
        status="idle",
        utility_weights={
            "cost": 0.3, "emissions": 0.2, "time": 0.3, "confidence": 0.2
        },
        current_time=simulation_start,
    ),
    VehicleAgent(
        id=1,
        vehicle_type="diesel",
        capacity=25,
        current_location=0,
        current_fuel=75,
        max_fuel=80,
        fuel_consumption_rate=0.08,
        emissions_factor=2.68,
        current_tasks=[],
        completed_tasks=[],
        status="idle",
        utility_weights={
            "cost": 0.4, "emissions": 0.1, "time": 0.4, "confidence": 0.1
        },
        current_time=simulation_start,
    ),
    VehicleAgent(
        id=2,
        vehicle_type="electric",
        capacity=20,
        current_location=0,
        current_fuel=100,
        max_fuel=100,
        fuel_consumption_rate=0.25,
        emissions_factor=0.0,
        current_tasks=[],
        completed_tasks=[],
        status="idle",
        utility_weights={
            "cost": 0.2, "emissions": 0.5, "time": 0.2, "confidence": 0.1
        },
        current_time=simulation_start,
    ),
    VehicleAgent(
        id=3,
        vehicle_type="electric",
        capacity=20,
        current_location=0,
        current_fuel=95,
        max_fuel=100,
        fuel_consumption_rate=0.25,
        emissions_factor=0.0,
        current_tasks=[],
        completed_tasks=[],
        status="idle",
        utility_weights={
            "cost": 0.3, "emissions": 0.4, "time": 0.2, "confidence": 0.1
        },
        current_time=simulation_start,
    ),
]

print(f"Created {len(agents)} autonomous vehicle agents:")
for agent in agents:
    energy_unit = "kWh" if agent.vehicle_type == "electric" else "L"
    print(
        f"  Agent {agent.id}: {agent.vehicle_type} van, "
        f"capacity {agent.capacity} kg, "
        f"energy {agent.current_fuel}/{agent.max_fuel} {energy_unit}"
    )
    print(f"    Utility weights: {agent.utility_weights}")

cnp = ContractNetProtocol(agents)
print(f"\nContract Net Protocol initialized with {len(agents)} agents.")
print("Ready for task announcements and bidding.")

In [ ]:
print("\n" + "=" * 60)
print("TASK CREATION FOR NEGOTIATION")
print("=" * 60)

tasks = [
    Task(
        id=1, task_type=TaskType.DELIVERY, location=1, demand=8,
        time_window=(8, 18), service_time=0.5, priority=2,
        deadline=simulation_start + timedelta(hours=8), announced_by=0
    ),
    Task(
        id=2, task_type=TaskType.DELIVERY, location=2, demand=12,
        time_window=(9, 17), service_time=0.4, priority=2,
        deadline=simulation_start + timedelta(hours=7), announced_by=1
    ),
    Task(
        id=3, task_type=TaskType.DELIVERY, location=3, demand=6,
        time_window=(10, 16), service_time=0.3, priority=3,
        deadline=simulation_start + timedelta(hours=6), announced_by=0
    ),
    Task(
        id=4, task_type=TaskType.DELIVERY, location=4, demand=10,
        time_window=(8, 18), service_time=0.6, priority=1,
        deadline=simulation_start + timedelta(hours=4), announced_by=2
    ),
    Task(
        id=5, task_type=TaskType.DELIVERY, location=5, demand=15,
        time_window=(11, 17), service_time=0.5, priority=2,
        deadline=simulation_start + timedelta(hours=5), announced_by=1
    ),
]

print(f"Created {len(tasks)} delivery tasks:")
for task in tasks:
    print(
        f"  Task {task.id}: location {task.location}, "
        f"demand {task.demand} kg, priority {task.priority}"
    )
    print(
        f"    Service window: {task.time_window[0]}:00-"
        f"{task.time_window[1]}:00; "
        f"deadline {task.deadline.strftime('%H:%M')}"
    )

print(f"\nTotal demand: {sum(task.demand for task in tasks)} kg")
print(f"Total fleet capacity: {sum(agent.capacity for agent in agents)} kg")

In [ ]:
print("\n" + "=" * 60)
print("MULTI-AGENT NEGOTIATION ROUND")
print("=" * 60)

results = cnp.run_negotiation_round(tasks)

print("\nNegotiation results:")
print(f"  Tasks announced: {results['tasks_announced']}")
print(f"  Contracts awarded: {results['contracts_awarded']}")
print(f"  Failed assignments: {results['failed_assignments']}")
print(f"  Estimated cost: ${results['total_cost']:.2f}")
print(f"  Estimated operational emissions: {results['total_emissions']:.2f} kg CO2")

print("\nAgent workload after the negotiation round:")
for agent_id, task_count in results["agent_utilization"].items():
    agent = cnp.agents_by_id[agent_id]
    assigned_demand = sum(task.demand for task in agent.current_tasks)
    print(
        f"  Agent {agent_id} ({agent.vehicle_type}): "
        f"{task_count} task(s), {assigned_demand} kg "
        f"of {agent.capacity} kg capacity"
    )

print("\nContract awards:")
for contract in cnp.contracts:
    winning_bid = next(
        bid for bid in contract.bids if bid.agent_id == contract.awarded_to
    )
    print(
        f"  Task {contract.id} awarded to Agent {contract.awarded_to}; "
        f"utility {winning_bid.utility_score:.3f}; "
        f"{len(contract.bids)} feasible bid(s)"
    )

assert results["contracts_awarded"] + results["failed_assignments"] == len(tasks)
assert all(
    sum(task.demand for task in agent.current_tasks) <= agent.capacity
    for agent in agents
)

In [ ]:
print("\n" + "=" * 60)
print("DYNAMIC EVENT SIMULATION")
print("=" * 60)

electric_agents = [
    agent
    for agent in agents
    if agent.vehicle_type == "electric" and agent.current_tasks
]
affected_agent = None

if electric_agents:
    affected_agent = electric_agents[0]
    failed_task = affected_agent.current_tasks.pop(0)
    affected_agent.current_fuel = 10.0
    affected_agent.status = "battery-limited"
    print(
        f"Agent {affected_agent.id} has a simulated battery constraint; "
        f"task {failed_task.id} requires reassignment."
    )
    cnp.handle_dynamic_reassignment(affected_agent.id, failed_task)
else:
    print("No electric agent has an assigned task to reassign in this run.")

print("\nFinal agent states:")
for agent in agents:
    energy_unit = "kWh" if agent.vehicle_type == "electric" else "L"
    print(
        f"  Agent {agent.id} ({agent.vehicle_type}): "
        f"{len(agent.current_tasks)} assigned task(s), "
        f"energy {agent.current_fuel:.1f}/{agent.max_fuel} {energy_unit}, "
        f"status {agent.status}, "
        f"simulated time {agent.current_time.strftime('%H:%M')}"
    )

In [ ]:
print("\n" + "=" * 60)
print("TASK DISTRIBUTION AND SYSTEM METRICS")
print("=" * 60)

task_distribution = {}
for agent in agents:
    demand = sum(task.demand for task in agent.current_tasks)
    task_distribution[agent.id] = {
        "vehicle_type": agent.vehicle_type,
        "num_tasks": len(agent.current_tasks),
        "total_demand": demand,
        "capacity_utilization": demand / agent.capacity,
        "utility_weights": agent.utility_weights,
    }

for agent_id, stats in task_distribution.items():
    print(
        f"Agent {agent_id} ({stats['vehicle_type']}): "
        f"{stats['num_tasks']} task(s), {stats['total_demand']:.1f} kg, "
        f"{stats['capacity_utilization']:.1%} capacity utilization"
    )

diesel_agents = [agent for agent in agents if agent.vehicle_type == "diesel"]
electric_agents_all = [
    agent for agent in agents if agent.vehicle_type == "electric"
]
diesel_load = sum(
    sum(task.demand for task in agent.current_tasks) for agent in diesel_agents
)
electric_load = sum(
    sum(task.demand for task in agent.current_tasks)
    for agent in electric_agents_all
)
diesel_capacity = sum(agent.capacity for agent in diesel_agents)
electric_capacity = sum(agent.capacity for agent in electric_agents_all)

print("\nLoad by vehicle type after the dynamic event:")
print(
    f"  Diesel: {diesel_load:.1f}/{diesel_capacity} kg "
    f"({diesel_load / diesel_capacity:.1%} capacity utilization)"
)
print(
    f"  Electric: {electric_load:.1f}/{electric_capacity} kg "
    f"({electric_load / electric_capacity:.1%} capacity utilization)"
)

avg_utilization = float(
    np.mean([stats["capacity_utilization"] for stats in task_distribution.values()])
)
task_completion_rate = results["contracts_awarded"] / len(tasks)
cost_per_awarded_task = results["total_cost"] / max(
    results["contracts_awarded"], 1
)
emissions_per_awarded_task = results["total_emissions"] / max(
    results["contracts_awarded"], 1
)
negotiation_summary = cnp.get_negotiation_summary()
total_bids = int(
negotiation_summary.loc[
    negotiation_summary["action"] == "task_announced", "bids_received"
].sum()
)
avg_bids_per_task = total_bids / len(tasks)

print("\nInitial negotiation-round metrics:")
print(f"  Task assignment rate: {task_completion_rate:.1%}")
print(f"  Mean agent capacity utilization: {avg_utilization:.1%}")
print(f"  Estimated cost per awarded task: ${cost_per_awarded_task:.2f}")
print(
    f"  Estimated operational emissions per awarded task: "
    f"{emissions_per_awarded_task:.2f} kg CO2"
)
print(f"  Feasible bids received: {total_bids}")
print(f"  Mean feasible bids per original task: {avg_bids_per_task:.1f}")

assert len(task_distribution) == len(agents)
assert all(
    0.0 <= stats["capacity_utilization"] <= 1.0
    for stats in task_distribution.values()
)

In [ ]:
from matplotlib.patches import Patch

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("Multi-Agent Negotiation: Run Summary", fontsize=16, fontweight="bold")

agent_ids = list(task_distribution)
vehicle_types = [task_distribution[agent_id]["vehicle_type"] for agent_id in agent_ids]
agent_colors = [
    "tab:red" if vehicle_type == "diesel" else "tab:green"
    for vehicle_type in vehicle_types
]

ax = axes[0, 0]
task_counts = [task_distribution[agent_id]["num_tasks"] for agent_id in agent_ids]
bars = ax.bar(agent_ids, task_counts, color=agent_colors, alpha=0.75)
ax.set(title="Assigned Tasks by Agent", xlabel="Agent ID", ylabel="Task count")
ax.grid(True, axis="y", alpha=0.3)
ax.legend(
    handles=[
        Patch(facecolor="tab:red", label="Diesel"),
        Patch(facecolor="tab:green", label="Electric"),
    ]
)
for bar, count in zip(bars, task_counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(count), ha="center", va="bottom")

ax = axes[0, 1]
utilizations = [
    100 * task_distribution[agent_id]["capacity_utilization"]
    for agent_id in agent_ids
]
bars = ax.bar(agent_ids, utilizations, color=agent_colors, alpha=0.75)
ax.set(title="Capacity Utilization by Agent", xlabel="Agent ID", ylabel="Capacity used (%)", ylim=(0, 110))
ax.grid(True, axis="y", alpha=0.3)
for bar, value in zip(bars, utilizations):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{value:.0f}%", ha="center", va="bottom")

ax = axes[0, 2]
categories = ["Cost", "Emissions", "Time", "Confidence"]
weight_data = np.array(
    [
        [
            task_distribution[agent_id]["utility_weights"][key]
            for agent_id in agent_ids
        ]
        for key in ("cost", "emissions", "time", "confidence")
    ]
)
x = np.arange(len(agent_ids))
width = 0.2
for index, category in enumerate(categories):
    ax.bar(x + index * width, weight_data[index], width, label=category)
ax.set(title="Agent Utility Weights", xlabel="Agent ID", ylabel="Weight")
ax.set_xticks(x + 1.5 * width, agent_ids)
ax.legend(fontsize=8)
ax.grid(True, axis="y", alpha=0.3)

ax = axes[1, 0]
types = ["Diesel", "Electric"]
loads = [diesel_load, electric_load]
capacities = [diesel_capacity, electric_capacity]
type_utilization = [100 * load / capacity for load, capacity in zip(loads, capacities)]
x = np.arange(len(types))
width = 0.35
ax.bar(x - width / 2, loads, width, label="Assigned demand (kg)")
ax.bar(x + width / 2, type_utilization, width, label="Capacity used (%)")
ax.set(title="Fleet Load by Vehicle Type", xlabel="Vehicle type", ylabel="Value")
ax.set_xticks(x, types)
ax.legend(fontsize=8)
ax.grid(True, axis="y", alpha=0.3)

ax = axes[1, 1]
timeline_data = negotiation_summary[
    negotiation_summary["action"].isin(["task_announced", "contract_awarded"])
].reset_index(drop=True)
if not timeline_data.empty:
    event_x = np.arange(len(timeline_data))
    event_colors = [
        "tab:blue" if action == "task_announced" else "tab:green"
        for action in timeline_data["action"]
    ]
    ax.scatter(event_x, np.zeros(len(event_x)), c=event_colors, s=65)
    for index, row in timeline_data.iterrows():
        if row["action"] == "task_announced":
            label = f"T{row['task_id']} announced"
        else:
            label = f"T{row['task_id']} awarded to A{row['awarded_to']}"
        ax.annotate(label, (index, 0), xytext=(0, 9), textcoords="offset points", rotation=45, ha="left", fontsize=7)
    ax.set_xlim(-1, len(event_x))
    ax.set_yticks([])
    ax.set_xlabel("Recorded event order")
    ax.grid(True, axis="x", alpha=0.3)
else:
    ax.text(0.5, 0.5, "No negotiation events", ha="center", va="center", transform=ax.transAxes)
ax.set_title("Negotiation Event Sequence")
ax.legend(
    handles=[
        Patch(facecolor="tab:blue", label="Task announced"),
        Patch(facecolor="tab:green", label="Contract awarded"),
    ],
    fontsize=8,
)

ax = axes[1, 2]
metric_labels = ["Task assignment", "Mean capacity use"]
metric_values = [100 * task_completion_rate, 100 * avg_utilization]
bars = ax.bar(metric_labels, metric_values, color=["skyblue", "lightgreen"])
ax.set(title="Initial-Round Summary Metrics", ylabel="Percentage (%)", ylim=(0, 110))
ax.grid(True, axis="y", alpha=0.3)
for bar, value in zip(bars, metric_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{value:.1f}%", ha="center", va="bottom")

fig.tight_layout()
plt.show()

In [ ]:
print("\n" + "=" * 60)
print("INTERPRETING THIS NEGOTIATION RUN")
print("=" * 60)

print("\n1. Task allocation:")
print(
    f"   The initial round awarded {results['contracts_awarded']} of "
    f"{len(tasks)} tasks. This outcome reflects the defined feasibility "
    "rules and agent utility weights."
)
print(
    f"   After the dynamic event, electric agents carry {electric_load:.1f} kg "
    f"and diesel agents carry {diesel_load:.1f} kg."
)

print("\n2. Negotiation activity:")
print(
    f"   The original tasks received {total_bids} feasible bids in total, "
    f"or {avg_bids_per_task:.1f} per task on average."
)
print(
    "   A higher utility score means a bid better matches that agent's "
    "weighted cost, emissions, time, and confidence criteria."
)

print("\n3. Dynamic reassignment:")
if affected_agent is None:
    print("   No electric agent had an assigned task, so no reassignment was tested.")
else:
    reassigned_task = next(
        (task for task in cnp.contracts if task.id == 1000 + failed_task.id),
        None,
    )
    if reassigned_task is None:
        print(
            "   The task was announced again, but no other agent submitted "
            "a feasible bid."
        )
    else:
        print(
            f"   Task {failed_task.id} was awarded again as task "
            f"{reassigned_task.id} to Agent {reassigned_task.awarded_to}."
        )

print("\n4. Scope of the evidence:")
print(
    "   This small deterministic example illustrates the CNP sequence; "
    "it does not establish global optimality, general scalability, or "
    "reliability under real-world uncertainty."
)
print(
    "   The cost and emissions totals describe only the initial negotiation "
    "round and use the simplified direct-leg model."
)

## Key lessons from multi-agent negotiation

The Contract Net Protocol separates task allocation into a sequence of announcements, bids, evaluations, and awards. Agents make local decisions using their own resource state and utility weights; the protocol coordinates those decisions by comparing bids for each announced task.

In this simulation, a bid is considered only when the agent has sufficient energy and capacity and can meet the task's service window and deadline. After an award, the agent's task load, location, available energy, and simulated time change. The reassignment event uses the same bidding process, so recovery is conditional on another agent being able to meet the task's constraints.

The example does not prove that the resulting allocation is globally optimal or that the system is fault tolerant in general. Its results depend on the chosen data, simplified travel model, announcement order, eligibility rules, and utility weights. The utility score expresses each agent's preferences; it is not a system-wide objective function.

### When this approach may be useful

Market-like negotiation can be useful when tasks and resources are distributed across autonomous decision makers and local adaptation matters. Its practical suitability depends on reliable communication, well-designed feasibility checks and utility functions, and rules for resolving unassigned tasks. The protocol also incurs communication and coordination costs, and its local awards do not by themselves provide a global optimality guarantee.

### Possible extensions

Natural extensions include additional operational constraints, richer travel-time and energy models, and larger test instances. Such extensions require validation of both the local bidding rules and the system-level behavior; they are outside the scope of this worked example.